# Lab 5B: Optimization drills
**Claude Certified Developer (Foundations) · 30-Hour Program for UST · Day 5 · about 20 minutes**

**Objectives**
- Trim a prompt without losing accuracy
- Cut output tokens safely
- Route between a cheap and a strong model by confidence
- Project monthly cost with caching and batching

**Before you start:** in Colab, click the key icon on the left, add a secret named `ANTHROPIC_API_KEY`, and turn on notebook access. All data in this lab is synthetic.

In [ ]:
# Setup: run once per session
!pip -q install -U anthropic 
import os, json, time, random, importlib, re
try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    assert os.environ.get("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY in your environment"
import anthropic
print("anthropic SDK", anthropic.__version__, "| key loaded:", bool(os.environ.get("ANTHROPIC_API_KEY")))

In [ ]:
%%writefile config.py
"""One place for Claude settings (same pattern as Day 1)."""
import os
# Confirm current IDs on the models overview page: https://platform.claude.com/docs/en/about-claude/models/overview
MODEL = os.environ.get("CLAUDE_MODEL", "claude-sonnet-5")
FAST_MODEL = os.environ.get("CLAUDE_FAST_MODEL", "claude-haiku-4-5-20251001")
MAX_TOKENS = 1024
MAX_RETRIES = 2
TIMEOUT_S = 120

In [ ]:
import config
importlib.reload(config)
client = anthropic.Anthropic(max_retries=config.MAX_RETRIES, timeout=config.TIMEOUT_S)

def log_response(resp, label=""):
    kinds = [b.type for b in resp.content]
    print(f"{label} stop_reason={resp.stop_reason} blocks={kinds} "
          f"input_tokens={resp.usage.input_tokens} output_tokens={resp.usage.output_tokens}")
    for b in resp.content:
        if b.type == "text":
            print("   ", b.text[:300].replace("\n", " "))

def text_of(resp):
    return "".join(b.text for b in resp.content if b.type == "text")

print("model:", config.MODEL, "| fast model:", config.FAST_MODEL)

In [ ]:
# Synthetic labelled tickets (training data only). Hard cases are marked with a star in the comments.
EVAL = [
 ("Whole sector 9 has had no fiber since 5am, a clinic is affected", "outage", "P1"),
 ("I was charged twice for September", "billing", "P2"),
 ("WiFi light blinks orange after a power cut", "device", "P3"),
 ("How do I change the email on my account?", "account", "P3"),
 ("Mobile data stopped working for everyone in our office block", "outage", "P1"),   # star: 'office' sounds like one customer
 ("Refund the router I returned last week, it's been 10 days", "billing", "P2"),
 ("Router keeps rebooting every hour since the firmware update", "device", "P2"),
 ("I'm locked out after too many password attempts", "account", "P2"),
 ("Is there a student discount?", "other", "P3"),
 ("Calls drop when I walk past the station, only on my phone", "device", "P3"),    # star: sounds like an outage
 ("Your invoice shows a roaming fee but I never left the city", "billing", "P2"),
 ("Can I transfer my plan to my brother?", "account", "P3"),
 ("Whole street lost TV and internet during the storm", "outage", "P1"),
 ("The app crashes when I open bill history", "other", "P3"),                      # star: billing words, app issue
 ("Please close my account at the end of the month", "account", "P3"),
 ("Speed test shows 5 Mbps on a 300 Mbps plan for a week", "device", "P2"),         # star: could be outage
]
print(len(EVAL), "labelled tickets")

In [ ]:
SCHEMA = {"type": "object", "properties": {
    "category": {"type": "string", "enum": ["billing", "outage", "device", "account", "other"]},
    "priority": {"type": "string", "enum": ["P1", "P2", "P3"]},
    "confidence": {"type": "number", "description": "0 to 1"}},
  "required": ["category", "priority", "confidence"], "additionalProperties": False}
SYSTEM = ("Classify telecom support tickets. outage = service down for many customers or an area. device = one customer's equipment or line. "
          "P1 = outage affecting many users or critical services; P2 = one customer blocked or charged wrongly; P3 = everything else.")
def classify(model, text, system=SYSTEM, max_tokens=200):
    t0 = time.perf_counter()
    r = client.messages.create(model=model, max_tokens=max_tokens, system=system,
                               output_config={"format": {"type": "json_schema", "schema": SCHEMA}},
                               messages=[{"role": "user", "content": text}])
    return json.loads(text_of(r)), time.perf_counter() - t0, r.usage

## Drill 1: trim the prompt
**Predict:** will the concise prompt lose accuracy?

In [ ]:
VERBOSE = SYSTEM + " " + ("Please read every ticket very carefully and think about the customer's situation before deciding. Remember that our customers are important "
                          "and that correct classification helps our support team respond quickly and accurately. ") * 6
def accuracy_and_tokens(system, model=None):
    rows = [classify(model or config.FAST_MODEL, t, system=system) for t, _, _ in EVAL]
    acc = np.mean([o["category"] == c for (o, _, _), (_, c, _) in zip(rows, EVAL)])
    return round(float(acc), 3), int(np.mean([u.input_tokens for _, _, u in rows]))
import numpy as np
for name, sys_prompt in (("verbose", VERBOSE), ("concise", SYSTEM)):
    acc, toks = accuracy_and_tokens(sys_prompt); print(f"{name:8s} accuracy={acc} avg_input_tokens={toks}")

## Drill 2: limit the output
Ask for only what you need. Then see what happens if max_tokens is too tight.

In [ ]:
q = "Customer says: 'charged twice for September'. Classify and explain."
free = client.messages.create(model=config.FAST_MODEL, max_tokens=600, system=SYSTEM, messages=[{"role": "user", "content": q}])
lean, _, lean_u = classify(config.FAST_MODEL, q)
print("free text output tokens:", free.usage.output_tokens, "| JSON-only output tokens:", lean_u.output_tokens)
tight = client.messages.create(model=config.FAST_MODEL, max_tokens=15, system=SYSTEM, messages=[{"role": "user", "content": q}])
print("tight max_tokens -> stop_reason:", tight.stop_reason, "| text:", text_of(tight))

Prefer a concise format over a tight max_tokens. A too-tight limit truncates; a format asks for less.

## Drill 3: route by confidence

In [ ]:
THRESHOLD = 0.8
def routed(text):
    out, _, u1 = classify(config.FAST_MODEL, text)
    if out["confidence"] >= THRESHOLD:
        return out, [(config.FAST_MODEL, u1)], False
    out2, _, u2 = classify(config.MODEL, text)
    return out2, [(config.FAST_MODEL, u1), (config.MODEL, u2)], True
rows = []
for text, cat, _ in EVAL:
    r_out, usages, escalated = routed(text)
    s_out, _, s_u = classify(config.MODEL, text)
    rows.append({"text": text[:45], "routed_ok": r_out["category"] == cat, "strong_only_ok": s_out["category"] == cat, "escalated": escalated,
                 "routed_usage": usages, "strong_usage": s_u})
rt = pd.DataFrame(rows)
print(f"routed accuracy {rt.routed_ok.mean():.0%} | strong-only accuracy {rt.strong_only_ok.mean():.0%} | escalation rate {rt.escalated.mean():.0%}")
rt[["text", "routed_ok", "strong_only_ok", "escalated"]]

In [ ]:
# Cost comparison: paste current prices (USD per million tokens) to see it
PRICES = {config.FAST_MODEL: (None, None), config.MODEL: (None, None)}
def usage_cost(model, u):
    p_in, p_out = PRICES[model]
    return None if p_in is None else (u.input_tokens * p_in + u.output_tokens * p_out) / 1e6
if all(v[0] is not None for v in PRICES.values()):
    routed_cost = sum(usage_cost(m, u) for us in rt.routed_usage for m, u in us)
    strong_cost = sum(usage_cost(config.MODEL, u) for u in rt.strong_usage)
    print(f"routed cost ${routed_cost:.5f} vs strong-only ${strong_cost:.5f} for {len(rt)} tickets")
else:
    print("Fill in prices to compare. Rule of thumb: routing pays when the escalation rate is low and escalations are accurate.")

## Drill 4: monthly projection

In [ ]:
REQUESTS_PER_DAY = 20000
AVG_IN = int(np.mean([u.input_tokens for u in rt.strong_usage])); AVG_OUT = int(np.mean([u.output_tokens for u in rt.strong_usage]))
P_IN, P_OUT = PRICES[config.MODEL]
BATCH_SHARE = 0.3          # share of traffic that can wait for batches (50% cheaper)
CACHED_PREFIX_TOKENS, CACHE_READ_X = 0, 0.1   # set a cached prefix size if you cache instructions
if P_IN is not None:
    base = (AVG_IN * P_IN + AVG_OUT * P_OUT) / 1e6
    cached = ((AVG_IN - CACHED_PREFIX_TOKENS) * P_IN + CACHED_PREFIX_TOKENS * P_IN * CACHE_READ_X + AVG_OUT * P_OUT) / 1e6
    monthly = REQUESTS_PER_DAY * 30 * cached * ((1 - BATCH_SHARE) + BATCH_SHARE * 0.5)
    print(f"per request ${base:.6f} -> with caching ${cached:.6f}; monthly with batching ${monthly:,.2f}")
else:
    print(f"Measured average: {AVG_IN} input and {AVG_OUT} output tokens per request. Add prices to project monthly cost.")

## You should now have
- [ ] Tokens saved by a concise prompt at equal accuracy
- [ ] Output tokens cut with a lean format
- [ ] Routing accuracy and escalation rate
- [ ] A monthly projection with your prices

## Check yourself
1. Which lever saved the most for this workload?
2. What threshold would you choose for routing, and how would you calibrate it?
3. What would you monitor after deploying routing?